# I-ADOPT benchmark demo notebook

Goal: show the **pipeline you actually use**:

1) Load **ground-truth JSON variables** from your test set folder  
2) Build the **LLM prompt** (instructions + schema + few-shot examples)  
3) Call the LLM and parse **JSON-only output** (robust extraction)  
4) Validate output against **JSON Schema**  
5) Wikidata linking using **cross-encoder only** 
6) Convert JSON → **RDF/Turtle (TTL)** 
7) Visualize TTL using **iadopt-vis** 


## 0) Imports & configuration

In [1]:
from __future__ import annotations

import json
import os
import pathlib
import re
from typing import Any, Dict, List, Optional

import requests
import numpy as np

# LLM client (OpenRouter via OpenAI client)
from openai import OpenAI, OpenAIError, APIStatusError
import httpx

# Cross-encoder reranker
from sentence_transformers import CrossEncoder

# Requires: %pip install jsonschema
from __future__ import annotations

import copy
from jsonschema import Draft202012Validator
import urllib.parse, webbrowser
from IPython.display import IFrame, display

/Users/rastegar-a/Documents/GitHub/i-adopt-llm-based-service/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ---- EDIT THESE PATHS TO MATCH YOUR MACHINE ----
DATA_DIR = pathlib.Path(
    "/Users/rastegar-a/Documents/GitHub/i-adopt-llm-based-service/benchmarking_example/data/Json_preferred/test_set"
)

# These are used for prompt building (optional, but recommended)
SCRIPT_DIR = pathlib.Path.cwd()
SCHEMA_PATH = SCRIPT_DIR / "data" / "Json_schema.json"
PROMPT_DIR = SCRIPT_DIR / "data" / "prompts"

# Model settings
MODEL_NAME = "qwen/qwen3.5-397b-a17b"
TEMPERATURE = 0.5

# OpenRouter key must be in your env
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_KEY = "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"

# Cross encoder model
CROSS_ENCODER_ID = "tomaarsen/Qwen3-Reranker-0.6B-seq-cls"
RERANK_DEVICE = "cpu"
RERANK_THRESHOLD = 0.8  # you can tune this

print("DATA_DIR exists:", DATA_DIR.exists())
print("SCHEMA_PATH exists:", SCHEMA_PATH.exists())
print("PROMPT_DIR exists:", PROMPT_DIR.exists())
print("OPENROUTER_API_KEY set:", bool(OPENROUTER_API_KEY))


DATA_DIR exists: True
SCHEMA_PATH exists: True
PROMPT_DIR exists: True
OPENROUTER_API_KEY set: True


## 1) Load input JSON variables (from your test set folder)

In [3]:
def load_gt_files_recursive(data_dir: pathlib.Path, max_vars: int = 0) -> List[tuple[pathlib.Path, Dict[str, Any]]]:
    paths = sorted(data_dir.rglob("*.json"))
    if max_vars and max_vars > 0:
        paths = paths[:max_vars]

    out: List[tuple[pathlib.Path, Dict[str, Any]]] = []
    for p in paths:
        out.append((p, json.loads(p.read_text(encoding="utf-8"))))
    return out

gt_items = load_gt_files_recursive(DATA_DIR, max_vars=100)  # limit for demo
print("Loaded:", len(gt_items))
print("Example file:", gt_items[41][0] if gt_items else None)


Loaded: 100
Example file: /Users/rastegar-a/Documents/GitHub/i-adopt-llm-based-service/benchmarking_example/data/Json_preferred/test_set/Challenge/C25_PolystyreneViscosity.json


In [4]:
# Pick one variable to demo
gt_path, gt = gt_items[41]
definition = gt.get("definition") or gt.get("comment") or ""
label = gt.get("label") or ""

print("Label:", label)
print("Definition:", definition[:300] + ("..." if len(definition) > 300 else ""))


Label: Dynamic shear viscosity of polystyrene PS042
Definition: Dynamic shear viscosity of polystyrene PS042 under the testing conditions of DIN 51810-1.


## 2) Build the LLM prompt

In [5]:
_EXAMPLE_HDR = "\n\n### Examples (valid against the same schema)\n"
_USER_HDR = "\n\n### Variable's definition to decompose\n"
_EXPECTED_HDR = "\n\n### Expected output\n*(only the JSON object)*"

def list_prompt_versions(prompt_dir: pathlib.Path) -> List[str]:
    if not prompt_dir.exists():
        return []
    return sorted(p.stem for p in prompt_dir.glob("*.txt"))

def load_prompt_instructions(prompt_dir: pathlib.Path, prompt_version: str) -> str:
    versions = list_prompt_versions(prompt_dir)
    if not versions:
        raise RuntimeError(f"No prompt templates found in {prompt_dir}")
    if not prompt_version:
        prompt_version = versions[0]
    if prompt_version not in versions:
        prompt_version = versions[0]
    return (prompt_dir / f"{prompt_version}.txt").read_text(encoding="utf-8").strip()

def strip_all_uri_fields(obj: Any) -> Any:
    """Remove keys containing 'URI' recursively (only for in-prompt examples)."""
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            if "URI" in k:
                continue
            if k.startswith("__"):
                continue
            out[k] = strip_all_uri_fields(v)
        return out
    if isinstance(obj, list):
        return [strip_all_uri_fields(x) for x in obj]
    return obj

def format_example_block(ex: Dict[str, Any], idx: int) -> str:
    definition = ex.get("definition") or ex.get("comment") or ""
    ex_no_uris = strip_all_uri_fields(ex)
    return (
        f"\n\n#### Example {idx}\n"
        f"{definition}\n\n"
        f"Expected output:\n{json.dumps(ex_no_uris, indent=2, ensure_ascii=False)}"
    )

FIVE_SHOT_DIR = SCRIPT_DIR / "data" / "Json_preferred" / "five_shot"

def load_examples(folder: pathlib.Path, n: int) -> List[Dict[str, Any]]:
    if n <= 0:
        return []
    paths = sorted(folder.glob("*.json"))
    return [json.loads(p.read_text(encoding="utf-8")) for p in paths[:n]]

def build_prompt(definition: str, prompt_version: str, examples: List[Dict[str, Any]] | None = None) -> str:
    examples = examples or []
    instructions = load_prompt_instructions(PROMPT_DIR, prompt_version)

    schema_text = SCHEMA_PATH.read_text(encoding="utf-8").strip() if SCHEMA_PATH.exists() else "{SCHEMA_PLACEHOLDER}"

    ex_block = ""
    if examples:
        blocks = [format_example_block(ex, i + 1) for i, ex in enumerate(examples)]
        ex_block = _EXAMPLE_HDR + "".join(blocks)

    return (
        f"{instructions}\n\n"
        f"### JSON-Schema\n{schema_text}\n"
        f"{ex_block}"
        f"{_USER_HDR}Variable's definition to decompose: {definition}"
        f"{_EXPECTED_HDR}"
    )
examples_5 = load_examples(FIVE_SHOT_DIR, 5)
prompt_versions = list_prompt_versions(PROMPT_DIR)
prompt_version = prompt_versions[0] if prompt_versions else ""
prompt = build_prompt(definition, prompt_version=prompt_version, examples=examples_5)

print("Prompt version:", prompt_version)
print(prompt)

Prompt version: constraint_tree
Follow the JSON-Schema exactly. Do not infer or invent new concepts.

definition must be exactly the same string as provided.
comment = short summary of the definition. Do not add new ideas.

hasProperty = the main measurable property in the definition.
hasObjectOfInterest = the thing that has this property.
hasMatrix = the medium in which the object occurs. Never a method or location.

If a required key is not in the definition, output an empty string for it.

Output only the JSON object.

Extraction order:

1. Copy definition exactly.
2. Extract hasProperty (main measurable characteristic).
3. Extract hasObjectOfInterest (entity with that property).
4. Extract hasMatrix only if the definition states a medium.
5. Extract hasConstraint last:
   • Only explicit limiting phrases.
   • label = short phrase
   • on = EXACT string from hasProperty or an entity
6. Never paraphrase or introduce new concepts.

hasConstraint rules (STRICT):

• Only include constr

## 3) Call the LLM + robust JSON extraction

In [6]:
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

_JSON_FENCE_RE = re.compile(r"```(?:json)?", re.MULTILINE)
_JSON_BLOCK_RE = re.compile(r"\{.*\}", re.DOTALL)

ONTO_KEYS = [
    "hasStatisticalModifier",
    "hasProperty",
    "hasObjectOfInterest",
    "hasMatrix",
    "hasContextObject",
    "hasConstraint",
]

def call_model(model: str, prompt: str, temperature: float) -> str:
    for attempt in range(1, 4):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=temperature,
                messages=[{"role": "user", "content": prompt}],
                timeout=60,
            )
            text = resp.choices[0].message.content or ""
            stripped = text.strip()
            if stripped.startswith("<!DOCTYPE html") or stripped.startswith("<html"):
                continue
            if not stripped:
                continue
            return text
        except APIStatusError as e:
            print(f"APIStatusError attempt {attempt}: {e}")
        except (OpenAIError, httpx.HTTPError) as e:
            print(f"Transport error attempt {attempt}: {e}")
        except Exception as e:
            print(f"Unexpected error attempt {attempt}: {e}")
    return ""

def coerce_prediction(pred: Dict[str, Any]) -> Dict[str, Any]:
    pred = dict(pred or {})
    for k in ONTO_KEYS:
        if k not in pred or pred[k] is None:
            pred[k] = [] if k == "hasConstraint" else ""
        elif k == "hasConstraint" and not isinstance(pred[k], list):
            pred[k] = []
    if isinstance(pred.get("hasProperty"), dict):
        pred["hasProperty"] = pred["hasProperty"].get("label", "") or ""
    return pred

def call_llm_loose(model: str, prompt: str, gt_label: str, definition: str, temperature: float) -> Dict[str, Any]:
    for attempt in range(1, 4):
        raw = call_model(model, prompt, temperature)
        if not raw.strip():
            continue
        cleaned = _JSON_FENCE_RE.sub("", raw).strip()
        m = _JSON_BLOCK_RE.search(cleaned)
        if not m:
            print("No JSON block found, attempt", attempt)
            continue
        try:
            data = json.loads(m.group(0))
        except Exception as e:
            print("JSON decode failure, attempt", attempt, e)
            continue

        data["label"] = gt_label
        data["definition"] = definition
        return coerce_prediction(data)
    return {}

# ⚠️ Running this cell will call the LLM.
# Uncomment to run:
pred = call_llm_loose(MODEL_NAME, prompt, gt_label=label, definition=definition, temperature=TEMPERATURE)

In [7]:
print("Ground Truth:\n", json.dumps(gt, indent=2, ensure_ascii=False))
print("Raw prediction:\n", json.dumps(pred, indent=2, ensure_ascii=False))

Ground Truth:
 {
  "label": "Dynamic shear viscosity of polystyrene PS042",
  "definition": "Dynamic shear viscosity of polystyrene PS042 under the testing conditions of DIN 51810-1.",
  "comment": "Dynamic shear viscosity of polystyrene PS042 under the testing conditions of DIN 51810-1.",
  "hasProperty": "dynamic shear viscosity",
  "hasPropertyURI": "https://www.wikidata.org/wiki/Q137640633",
  "hasObjectOfInterest": "polystyrene PS042",
  "hasConstraint": [
    {
      "label": "condition: under testing conditions of DIN 5810-1",
      "on": "dynamic shear viscosity"
    }
  ]
}
Raw prediction:
 {
  "label": "Dynamic shear viscosity of polystyrene PS042",
  "definition": "Dynamic shear viscosity of polystyrene PS042 under the testing conditions of DIN 51810-1.",
  "comment": "Dynamic shear viscosity of polystyrene PS042 under testing conditions DIN 51810-1.",
  "hasProperty": "dynamic viscosity",
  "hasObjectOfInterest": "polystyrene",
  "hasConstraint": [
    {
      "label": "typ

## 4) JSON Schema validation


- load schema (already available at `SCHEMA_PATH`)
- validate prediction
- if invalid → show errors, optionally re-prompt


In [8]:
# --- JSON Schema validation ---
def _format_path(err) -> str:
    """Convert jsonschema error path to a readable dotted path."""
    if not err.path:
        return "$"
    out = "$"
    for p in err.path:
        if isinstance(p, int):
            out += f"[{p}]"
        else:
            out += f".{p}"
    return out


def _safe_preview(value: Any, limit: int = 200) -> str:
    """Small readable preview of the offending value."""
    try:
        s = json.dumps(value, ensure_ascii=False)
    except Exception:
        s = repr(value)
    if len(s) > limit:
        s = s[:limit] + "…"
    return s


def _patch_schema_for_pipeline(schema: Dict[str, Any]) -> Dict[str, Any]:
    """
    Pipeline-specific compatibility patch:

    You said hasConstraint is optional and can be an empty list: "hasConstraint": []
    But your schema currently contains: "minItems": 1, which would reject [].

    We patch minItems to 0 *at runtime* to match your pipeline semantics.
    (Alternatively, edit the schema file and set minItems to 0.)
    """
    patched = copy.deepcopy(schema)

    try:
        hc = patched["properties"]["hasConstraint"]
        # if it's present and minItems is too strict, relax it
        if isinstance(hc, dict) and hc.get("minItems", None) == 1:
            hc["minItems"] = 0
    except Exception:
        # If schema structure changes later, do nothing
        pass

    return patched


def load_schema(schema_path) -> Dict[str, Any]:
    return json.loads(schema_path.read_text(encoding="utf-8"))


def validate_against_schema(
    instance: Dict[str, Any],
    *,
    schema_path=SCHEMA_PATH,
    schema: Optional[Dict[str, Any]] = None,
    label_for_logs: Optional[str] = None,
) -> None:
    """
    Validate a model output dict against the JSON Schema.

    - Loads schema from schema_path (unless schema provided).
    - Applies a small runtime patch (hasConstraint minItems) to match pipeline behavior.
    - Raises ValueError with clear, path-specific messages.

    label_for_logs: optional variable label used in error headers.
    """
    if schema is None:
        schema = load_schema(schema_path)

    schema = _patch_schema_for_pipeline(schema)

    validator = Draft202012Validator(schema)
    errors = sorted(validator.iter_errors(instance), key=lambda e: list(e.path))

    if not errors:
        return

    header = f"Schema validation failed"
    if label_for_logs:
        header += f" for variable: {label_for_logs}"

    lines: List[str] = [header, "-" * len(header)]

    # show up to N errors to keep output readable
    MAX_ERRS = 30
    for i, err in enumerate(errors[:MAX_ERRS], start=1):
        path = _format_path(err)
        offending_value = _safe_preview(err.instance)
        lines.append(f"{i:02d}) Path: {path}")
        lines.append(f"    Error: {err.message}")
        lines.append(f"    Offending value: {offending_value}")

        # Extra hint: if it failed inside entityOrSystem, say so explicitly
        # (helps users understand whether failure is in hasMatrix/hasOOI/hasContextObject)
        if path.startswith("$.hasObjectOfInterest") or path.startswith("$.hasMatrix") or path.startswith("$.hasContextObject"):
            lines.append("    Hint: This error is inside an entityOrSystem field (string vs AsymmetricSystem vs SymmetricSystem).")

    if len(errors) > MAX_ERRS:
        lines.append(f"... plus {len(errors) - MAX_ERRS} more errors.")

    raise ValueError("\n".join(lines))


# ---- Example wiring (use pred from the LLM step) ----
# pred = call_llm_loose(...)
try:
    validate_against_schema(pred, label_for_logs=pred.get("label"))
    print("✅ Schema-valid")
except ValueError as e:
    print(str(e))


✅ Schema-valid


## 5) Wikidata linking (Phase 3) — **cross-encoder only**

In [9]:
# --- Qwen3 reranker formatting ---
QWEN3_RERANK_PREFIX = (
    "<|im_start|>system\n"
    " Judge whether the Document meets the requirements based on the Query and the Instruct provided. "
    'Note that the answer can only be "yes" or "no".<|im_end|>\n'
    "<|im_start|>user\n"
)
QWEN3_RERANK_SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

DEFAULT_RERANK_TASK = "Given a web search query, retrieve relevant passages that answer the query"

def format_queries(query: str, task: str = DEFAULT_RERANK_TASK) -> str:
    return f"{QWEN3_RERANK_PREFIX}<Instruct>: {task}\n<Query>: {query}\n"

def format_document(doc: str) -> str:
    return f"<Document>: {doc}{QWEN3_RERANK_SUFFIX}"

def _qid_from_uri_or_text(s: Optional[str]) -> Optional[str]:
    if not s:
        return None
    m = re.search(r"(Q\d+)", s)
    return m.group(1) if m else None

def _to_wiki_url(uri: Optional[str]) -> Optional[str]:
    if not uri:
        return None
    q = _qid_from_uri_or_text(uri)
    return f"https://www.wikidata.org/wiki/{q}" if q else uri.strip().replace("http://", "https://")

# Cross-encoder model load
reranker = CrossEncoder(CROSS_ENCODER_ID, device=RERANK_DEVICE)

def get_wikidata_entity_cross_encoder(
    term: str,
    context: str = "",
    threshold: float = RERANK_THRESHOLD,
) -> Optional[str]:
    """Wikidata entity resolution using cross-encoder reranking only."""
    if not term:
        return None

    encoded = urllib.parse.quote_plus(term)
    headers = {"User-Agent": "IADOPT-Linker/1.0 (+notebook)"}
    url = f"https://www.wikidata.org/w/api.php?action=wbsearchentities&search={encoded}&language=en&format=json"

    r = requests.get(url, headers=headers, timeout=20)
    if r.status_code != 200:
        return None

    search = r.json().get("search", [])
    if not search:
        return None

    query = f'Definition of "{term}" in context: "{context}"'
    documents = [f'label: "{s.get("label","")}", description: "{s.get("description","")}"' for s in search]
    pairs = [[format_queries(query, DEFAULT_RERANK_TASK), format_document(doc)] for doc in documents]
    scores = reranker.predict(pairs, show_progress_bar=False)

    ranked = sorted(zip(search, scores), key=lambda x: float(x[1]), reverse=True)
    best_s, best_score = ranked[0]

    return _to_wiki_url(best_s["id"]) if float(best_score) >= float(threshold) else None

def enrich_with_uris_cross_encoder(pred: Dict[str, Any], threshold: float = RERANK_THRESHOLD) -> Dict[str, Any]:
    """Attach ...URI fields using cross-encoder ranking (only)."""
    out = json.loads(json.dumps(pred))  # deep copy

    def add_uri_field(container: Dict[str, Any], key: str, label_value: Any):
        if isinstance(label_value, str) and label_value.strip():
            uri = get_wikidata_entity_cross_encoder(
                label_value,
                context=pred.get("definition", ""),
                threshold=threshold,
            )
            if uri:
                container[f"{key}URI"] = _to_wiki_url(uri)

    # top-level strings
    for p in ["hasProperty", "hasMatrix", "hasObjectOfInterest", "hasContextObject", "hasStatisticalModifier"]:
        if p in out and isinstance(out[p], str):
            add_uri_field(out, p, out[p])

    # nested systems (AsymmetricSystem / SymmetricSystem)
    for p in ["hasMatrix", "hasObjectOfInterest", "hasContextObject"]:
        val = out.get(p)
        if isinstance(val, dict):
            if "AsymmetricSystem" in val:
                for kk in ["AsymmetricSystem", "hasSource", "hasTarget"]:
                    if val.get(kk):
                        uri = get_wikidata_entity_cross_encoder(
                            val[kk],
                            context=pred.get("definition", ""),
                            threshold=threshold,
                        )
                        if uri:
                            val[f"{kk}URI"] = _to_wiki_url(uri)

            if "SymmetricSystem" in val:
                if val.get("SymmetricSystem"):
                    uri = get_wikidata_entity_cross_encoder(
                        val["SymmetricSystem"],
                        context=pred.get("definition", ""),
                        threshold=threshold,
                    )
                    if uri:
                        val["SymmetricSystemURI"] = _to_wiki_url(uri)

                parts = val.get("hasPart", [])
                if isinstance(parts, list) and parts:
                    part_uris = []
                    for part in parts:
                        if isinstance(part, str) and part.strip():
                            uri = get_wikidata_entity_cross_encoder(
                                part,
                                context=pred.get("definition", ""),
                                threshold=threshold,
                            )
                            part_uris.append(_to_wiki_url(uri) if uri else None)
                        else:
                            part_uris.append(None)
                    if any(part_uris):
                        val["hasPartURIs"] = part_uris

    return out


In [10]:
# Example wiring (requires pred from LLM step):
pred_enriched = enrich_with_uris_cross_encoder(pred, threshold=RERANK_THRESHOLD)
print("Ground Truth:\n", json.dumps(gt, indent=2, ensure_ascii=False))
print("Enriched prediction:\n", json.dumps(pred_enriched, indent=2, ensure_ascii=False))

Ground Truth:
 {
  "label": "Dynamic shear viscosity of polystyrene PS042",
  "definition": "Dynamic shear viscosity of polystyrene PS042 under the testing conditions of DIN 51810-1.",
  "comment": "Dynamic shear viscosity of polystyrene PS042 under the testing conditions of DIN 51810-1.",
  "hasProperty": "dynamic shear viscosity",
  "hasPropertyURI": "https://www.wikidata.org/wiki/Q137640633",
  "hasObjectOfInterest": "polystyrene PS042",
  "hasConstraint": [
    {
      "label": "condition: under testing conditions of DIN 5810-1",
      "on": "dynamic shear viscosity"
    }
  ]
}
Enriched prediction:
 {
  "label": "Dynamic shear viscosity of polystyrene PS042",
  "definition": "Dynamic shear viscosity of polystyrene PS042 under the testing conditions of DIN 51810-1.",
  "comment": "Dynamic shear viscosity of polystyrene PS042 under testing conditions DIN 51810-1.",
  "hasProperty": "dynamic viscosity",
  "hasObjectOfInterest": "polystyrene",
  "hasConstraint": [
    {
      "label":

## 6) JSON → TTL 


What this step must do:
- Create an `iop:Variable` node
- Add `rdfs:label` and `skos:definition`/`rdfs:comment`
- For each I-ADOPT component:
  - Create nodes for `hasProperty`, `hasMatrix`, `hasObjectOfInterest`, `hasContextObject`, `hasStatisticalModifier`
  - If `...URI` exists (Wikidata), use it as the subject URI (typically `https://www.wikidata.org/entity/Q...`)
  - Otherwise mint a stable local URI (e.g., `http://example.org/...`)
- For systems:
  - `AsymmetricSystem`: create node of `iop:AsymmetricSystem` and add `iop:hasSource`, `iop:hasTarget`
  - `SymmetricSystem`: create node of `iop:SymmetricSystem` and add `iop:hasPart` (1..n)
- For constraints:
  - Create `iop:Constraint` blank nodes or minted nodes
  - Add `rdfs:label` for the constraint text
  - Add `iop:constrains` pointing to the target component indicated by `on`

Output:
- A Turtle string (TTL)
- Optionally write to file, one TTL per variable JSON


In [11]:
from __future__ import annotations

import re
from typing import Any, Dict, Optional, Tuple, List


# ---- Fixed prefixes to match your repo style ----
PREFIXES = """@prefix iop: <https://w3id.org/iadopt/ont/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix ex: <http://example.org/iadopt/> .
@prefix iopp: <https://w3id.org/iadopt/pattern/> .
@prefix patt: <http://example.org/iadopt/pattern> .
@prefix sosa: <http://www.w3.org/ns/sosa/> .
@prefix uom: <http://www.ontology-of-units-of-measure.org/resource/om-2/> .

"""

WIKIDATA_WIKI = "https://www.wikidata.org/wiki/"
WIKIDATA_ENTITY = "https://www.wikidata.org/entity/"


def wiki_to_entity(uri: Optional[str]) -> Optional[str]:
    """https://www.wikidata.org/wiki/Q123 -> https://www.wikidata.org/entity/Q123"""
    if not uri:
        return None
    m = re.search(r"(Q\d+)", uri)
    if not m:
        return None
    return WIKIDATA_ENTITY + m.group(1)


def _camel_name_from_label(label: str) -> str:
    """
    ex:ActinicFlux style name.
    Heuristic: remove non-alphanumerics, title-case tokens, drop very short stopwords.
    """
    label = (label or "").strip()
    if not label:
        return "Variable"

    # keep alnum, split into tokens
    tokens = re.findall(r"[A-Za-z0-9]+", label)
    if not tokens:
        return "Variable"

    stop = {"of", "the", "and", "in", "on", "at", "for", "to", "a", "an"}
    cleaned = [t for t in tokens if t.lower() not in stop]

    # In your example, they used ActinicFlux not "SpectralUpwellingActinicFlux"
    # We'll bias toward "main noun phrase" by dropping leading adjectives like spectral/upwelling if many tokens.
    # (You can adjust this rule later.)
    if len(cleaned) >= 3:
        # drop first token if it's a common adjective-ish word
        drop = {"spectral", "upwelling", "downwelling", "vertical", "horizontal"}
        if cleaned[0].lower() in drop:
            cleaned = cleaned[1:]

    return "".join(t[:1].upper() + t[1:] for t in cleaned)


def _ttl_quote_multiline(text: str) -> str:
    """Triple-quote string like your TTL style."""
    text = text or ""
    # avoid accidental """ inside
    text = text.replace('"""', '\\"""')
    return f'"""{text}"""'


def _indent(lines: str, n: int = 4) -> str:
    pad = " " * n
    return "\n".join(pad + ln if ln.strip() else ln for ln in lines.splitlines())


def _entity_block(uri: str, rdf_type: str, label: str) -> str:
    """
    <https://www.wikidata.org/entity/Q...>
        a iop:Entity ;
        rdfs:label "..." .
    """
    return f"""<{uri}>
    a 
        {rdf_type} ;
    rdfs:label 
        "{label}" .

"""


def json_to_ttl_repo_style(
    pred: Dict[str, Any],
    *,
    issue_url: Optional[str] = None,
    ex_base: str = "http://example.org/iadopt/",
) -> str:
    """
    Convert enriched prediction JSON -> TTL matching the exact style in your example.

    Rules:
    - Variable subject is ex:<CamelName> derived from variable label.
    - If ...URI exists -> use Wikidata entity URI in triples + create bottom blocks.
    - If ...URI missing but label exists -> mint ex:<CamelName> node and still type+label it.
    - hasConstraint is allowed to be [] (no constraints).
    - Constraint 'on' resolves to:
        * a component label match (property/matrix/ooi/context/statmod)
        * else, if it already looks like QID/URI, use it
        * else, skip constrains target fallback to variable (or keep as literal — but we’ll fallback to variable URI)
    """
    label = (pred.get("label") or "").strip()
    definition = (pred.get("definition") or "").strip()
    comment = (pred.get("comment") or "").strip()

    var_name = _camel_name_from_label(label)
    var_subject = f"ex:{var_name}"

    # Collect component nodes: key -> (uri, type, label)
    # Types per your example: Property -> iop:Property, others -> iop:Entity, statmod -> iop:StatisticalModifier
    components: Dict[str, Tuple[str, str, str]] = {}

    def add_component(field: str, rdf_type: str) -> Optional[str]:
        val = pred.get(field)
        if not isinstance(val, str) or not val.strip():
            return None
        val = val.strip()

        uri_field = f"{field}URI"
        wd_entity = wiki_to_entity(pred.get(uri_field))

        if wd_entity:
            uri = wd_entity
        else:
            # mint local URI in ex namespace, but still consistent looking
            local = _camel_name_from_label(val)
            uri = ex_base.rstrip("/") + "/" + local  # e.g., http://example.org/iadopt/Atmosphere

        components[field] = (uri, rdf_type, val)
        return uri

    # Add the fields you use
    prop_uri = add_component("hasProperty", "iop:Property")
    ooi_uri  = add_component("hasObjectOfInterest", "iop:Entity")
    mat_uri  = add_component("hasMatrix", "iop:Entity")
    ctx_uri  = add_component("hasContextObject", "iop:Entity")
    stat_uri = add_component("hasStatisticalModifier", "iop:StatisticalModifier")

    # Helper: resolve constraint target by label
    label_to_uri = {components[k][2].lower(): components[k][0] for k in components}

    def resolve_on_target(on_text: str) -> str:
        on_text = (on_text or "").strip()
        if not on_text:
            # fallback to variable itself
            return var_subject

        # allow "hasProperty: X" style
        on_clean = re.sub(r"^\s*[A-Za-z][A-Za-z0-9_]*\s*:\s*", "", on_text).strip()

        # match by label
        if on_text.lower() in label_to_uri:
            return f"<{label_to_uri[on_text.lower()]}>"
        if on_clean.lower() in label_to_uri:
            return f"<{label_to_uri[on_clean.lower()]}>"

        # if looks like a wikidata QID
        m = re.search(r"(Q\d+)", on_text)
        if m:
            return f"<{WIKIDATA_ENTITY}{m.group(1)}>"
        if on_text.startswith("http://") or on_text.startswith("https://"):
            # normalize to entity if it's wikidata/wiki
            wd = wiki_to_entity(on_text)
            return f"<{wd}>" if wd else f"<{on_text}>"

        # last fallback: variable itself
        return var_subject

    # Build constraint list text
    constraints = pred.get("hasConstraint") or []
    constraint_blocks: List[str] = []
    if isinstance(constraints, list):
        for c in constraints:
            if not isinstance(c, dict):
                continue
            c_label = (c.get("label") or "").strip()
            c_on = (c.get("on") or "").strip()
            if not c_label:
                continue
            target = resolve_on_target(c_on)

            constraint_blocks.append(
                f"""[ a iop:Constraint ;
             rdfs:label "{c_label}" ;
             iop:constrains {target} ;
        ]"""
            )

    # Join constraints in the same comma-separated style
    has_constraint_line = ""
    if constraint_blocks:
        joined = " ,\n        ".join(constraint_blocks)
        has_constraint_line = f"    iop:hasConstraint \n        {joined} .\n"
    else:
        # If no constraints, end the variable block with "."
        has_constraint_line = "    .\n"

    # Variable block
    var_lines = [
        f"{var_subject}",
        "    a ",
        "        iop:Variable ;",
        "    rdfs:label ",
        f'        "{label}" ;',
        "    skos:definition ",
        f"        {_ttl_quote_multiline(definition)} ;",
        "    rdfs:comment ",
        f"        {_ttl_quote_multiline(comment)} ;",
    ]

    if issue_url:
        var_lines.append(f'    ex:issue "{issue_url}" ;')

    # Add iop:has... predicates for present components (match ordering in your example)
    if ooi_uri:
        var_lines.append(f"    iop:hasObjectOfInterest \n        <{ooi_uri}> ;")
    if mat_uri:
        var_lines.append(f"    iop:hasMatrix \n        <{mat_uri}> ;")
    if prop_uri:
        var_lines.append(f"    iop:hasProperty \n        <{prop_uri}> ;")
    if ctx_uri:
        var_lines.append(f"    iop:hasContextObject \n        <{ctx_uri}> ;")
    if stat_uri:
        var_lines.append(f"    iop:hasStatisticalModifier \n        <{stat_uri}> ;")

    # Replace last ";" with ";" (we will finish via has_constraint_line)
    var_block = "\n".join(var_lines) + "\n" + has_constraint_line + "\n"

    # Bottom blocks for referenced URIs (exactly like your example)
    bottom = ""
    # Only emit bottom blocks for URIs we referenced in the variable block or constraints.
    # (In your example they do for property/matrix/ooi.)
    for field in ["hasObjectOfInterest", "hasMatrix", "hasProperty", "hasContextObject", "hasStatisticalModifier"]:
        if field in components:
            uri, rdf_type, lbl = components[field]
            bottom += _entity_block(uri, rdf_type, lbl)

    return PREFIXES + var_block + bottom


In [12]:
ttl = json_to_ttl_repo_style(
    pred_enriched,
    issue_url=None  # or set if you have it
)

print(ttl)


@prefix iop: <https://w3id.org/iadopt/ont/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix skos: <http://www.w3.org/2004/02/skos/core#> .
@prefix ex: <http://example.org/iadopt/> .
@prefix iopp: <https://w3id.org/iadopt/pattern/> .
@prefix patt: <http://example.org/iadopt/pattern> .
@prefix sosa: <http://www.w3.org/ns/sosa/> .
@prefix uom: <http://www.ontology-of-units-of-measure.org/resource/om-2/> .

ex:DynamicShearViscosityPolystyrenePS042
    a 
        iop:Variable ;
    rdfs:label 
        "Dynamic shear viscosity of polystyrene PS042" ;
    skos:definition 
        """Dynamic shear viscosity of polystyrene PS042 under the testing conditions of DIN 51810-1.""" ;
    rdfs:comment 
        """Dynamic shear viscosity of polystyrene PS042 under testing conditions DIN 51810-1.""" ;
    iop:hasObjectOfInterest 
        <https://www.wikidata.org/entity/Q146243> ;
    iop:hasProperty 
        <https://www.wik

## 7) Visualization with iadopt-vis 


What this step must do:
- Take the produced TTL
- Feed it into the iadopt-vis workflow 
- Produce a visualization (SVG/HTML) or open the UI with preloaded data


In [13]:
def open_iadopt_vis_interactive(ttl_text: str, base="http://localhost:5173/", embed=True):
    url = f"{base}?ttl={urllib.parse.quote(ttl_text, safe='')}"
    webbrowser.open(url, new=2)
    if embed:
        display(IFrame(url, width="100%", height=900))
    return url

# usage:
# open_iadopt_vis_interactive(ttl)



In [14]:
url = open_iadopt_vis_interactive(ttl, embed=True)
print("Opened:", url)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Opened: http://localhost:5173/?ttl=%40prefix%20iop%3A%20%3Chttps%3A%2F%2Fw3id.org%2Fiadopt%2Font%2F%3E%20.%0A%40prefix%20rdf%3A%20%3Chttp%3A%2F%2Fwww.w3.org%2F1999%2F02%2F22-rdf-syntax-ns%23%3E%20.%0A%40prefix%20rdfs%3A%20%3Chttp%3A%2F%2Fwww.w3.org%2F2000%2F01%2Frdf-schema%23%3E%20.%0A%40prefix%20skos%3A%20%3Chttp%3A%2F%2Fwww.w3.org%2F2004%2F02%2Fskos%2Fcore%23%3E%20.%0A%40prefix%20ex%3A%20%3Chttp%3A%2F%2Fexample.org%2Fiadopt%2F%3E%20.%0A%40prefix%20iopp%3A%20%3Chttps%3A%2F%2Fw3id.org%2Fiadopt%2Fpattern%2F%3E%20.%0A%40prefix%20patt%3A%20%3Chttp%3A%2F%2Fexample.org%2Fiadopt%2Fpattern%3E%20.%0A%40prefix%20sosa%3A%20%3Chttp%3A%2F%2Fwww.w3.org%2Fns%2Fsosa%2F%3E%20.%0A%40prefix%20uom%3A%20%3Chttp%3A%2F%2Fwww.ontology-of-units-of-measure.org%2Fresource%2Fom-2%2F%3E%20.%0A%0Aex%3ADynamicShearViscosityPolystyrenePS042%0A%20%20%20%20a%20%0A%20%20%20%20%20%20%20%20iop%3AVariable%20%3B%0A%20%20%20%20rdfs%3Alabel%20%0A%20%20%20%20%20%20%20%20%22Dynamic%20shear%20viscosity%20of%20polystyrene%20PS04